In [1]:
# ============================================================
# PHASE 4 — Cell 1: Setup + Load Standard Split
# ============================================================
!pip install wfdb -q

from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# Load standard split NPZ
train = np.load(os.path.join(PROCESSED_DIR, 'train.npz'), allow_pickle=True)
val = np.load(os.path.join(PROCESSED_DIR, 'val.npz'), allow_pickle=True)
test = np.load(os.path.join(PROCESSED_DIR, 'test.npz'), allow_pickle=True)

X_train = train['X']; y_train = train['y']; R_train = train['R']; rec_train = train['record_id']
X_val = val['X']; y_val = val['y']; R_val = val['R']; rec_val = val['record_id']
X_test = test['X']; y_test = test['y']; R_test = test['R']; rec_test = test['record_id']

print(f"\n✅ Loaded standard split data")
print(f"   Train: X={X_train.shape}, R={R_train.shape}")
print(f"   Val:   X={X_val.shape}, R={R_val.shape}")
print(f"   Test:  X={X_test.shape}, R={R_test.shape}")

# Check class distribution
print(f"\nClass distribution (original AAMI 5-class):")
print(f"  Train: {dict(Counter(y_train))}")
print(f"  Val:   {dict(Counter(y_val))}")
print(f"  Test:  {dict(Counter(y_test))}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.8 MB/s eta 0:00:00
Mounted at /content/drive
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

✅ Loaded standard split data
   Train: X=(44728, 259), R=(44728, 4)
   Val:   X=(12822, 259), R=(12822, 4)
   Test:  X=(51899, 259), R=(51899, 4)

Class distribution (original AAMI 5-class):
  Train: {np.int64(0): 37176, np.int64(4): 4151, np.int64(1): 559, np.int64(2): 2800, np.int64(3): 42}
  Val:   {np.int64(4): 2079, np.int64(2): 1053, np.int64(0): 8933, np.int64(3): 372, np.int64(1): 385}
  Test:  {np.int64(0): 44483, np.int64(1): 1837, np.int64(2): 3382, np.int64(4): 1809, np.int64(3): 388}


In [2]:
# ============================================================
# PHASE 4 — Cell 2: Apply NS_V_Q Labels
# ============================================================
# Original AAMI 5-class:
# 0=N, 1=S, 2=V, 3=F, 4=Q
# NS_V_Q mapping:
# 0 (N) → 0 (N/S)
# 1 (S) → 0 (N/S)
# 2 (V) → 1 (V)
# 3 (F) → DROP
# 4 (Q) → 2 (Q)

LABEL_MAP = {0: 0, 1: 0, 2: 1, 4: 2}
CLASS_NAMES = ['N/S', 'V', 'Q']


def apply_nsvq(X, y, R, rec):
    """Apply NS_V_Q mapping: drop F, merge S with N."""
    mask = y != 3  # Drop F-class
    X2 = X[mask]
    y2 = y[mask].copy()
    R2 = R[mask]
    rec2 = rec[mask]

    # Apply mapping
    new_y = np.zeros_like(y2)
    for old, new in LABEL_MAP.items():
        new_y[y2 == old] = new
    return X2, new_y, R2, rec2


X_train, y_train, R_train, rec_train = apply_nsvq(X_train, y_train, R_train, rec_train)
X_val, y_val, R_val, rec_val = apply_nsvq(X_val, y_val, R_val, rec_val)
X_test, y_test, R_test, rec_test = apply_nsvq(X_test, y_test, R_test, rec_test)

print(f"✅ Applied NS_V_Q labels")
print(f"   Classes: {CLASS_NAMES}")
print(f"\nNew class distribution:")
print(f"  Train: {dict(Counter(y_train))}")
print(f"  Val:   {dict(Counter(y_val))}")
print(f"  Test:  {dict(Counter(y_test))}")

print(f"\n✅ Data ready for training")

✅ Applied NS_V_Q labels
   Classes: ['N/S', 'V', 'Q']

New class distribution:
  Train: {np.int64(0): 37735, np.int64(2): 4151, np.int64(1): 2800}
  Val:   {np.int64(2): 2079, np.int64(1): 1053, np.int64(0): 9318}
  Test:  {np.int64(0): 46320, np.int64(1): 3382, np.int64(2): 1809}

✅ Data ready for training


In [3]:
# ============================================================
# PHASE 4 — Cell 3: Build Two-Branch Model
# ============================================================
def focal_loss(gamma=2.0, num_classes=3):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_oh = tf.one_hot(y_true, num_classes)
        p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        focal_weight = tf.pow(1.0 - p_t, gamma)
        ce = -tf.math.log(p_t)
        return tf.reduce_mean(focal_weight * ce)
    return loss_fn


def build_two_branch_model(beat_len, n_rr_features=4, n_classes=3, seed=42):
    tf.keras.utils.set_random_seed(seed)
    reg = tf.keras.regularizers.l2(1e-4)

    # Branch 1: Waveform (CNN)
    wave_in = layers.Input(shape=(beat_len, 1), name='waveform')
    x = layers.Conv1D(64, 7, activation='relu', padding='same', kernel_regularizer=reg)(wave_in)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, 5, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(256, 3, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    wave_feat = layers.GlobalAveragePooling1D()(x)

    # Branch 2: RR features (Dense)
    rr_in = layers.Input(shape=(n_rr_features,), name='rr_features')
    r = layers.Dense(32, activation='relu', kernel_regularizer=reg)(rr_in)
    r = layers.BatchNormalization()(r)
    r = layers.Dense(16, activation='relu', kernel_regularizer=reg)(r)
    rr_feat = r

    # Merge
    merged = layers.Concatenate()([wave_feat, rr_feat])
    d = layers.Dense(64, activation='relu', kernel_regularizer=reg)(merged)
    d = layers.Dropout(0.4)(d)
    out = layers.Dense(n_classes, activation='softmax')(d)

    model = Model(inputs=[wave_in, rr_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss=focal_loss(gamma=2.0, num_classes=3),
        metrics=['accuracy']
    )
    return model


beat_len = X_train.shape[1]
model = build_two_branch_model(beat_len)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ waveform            │ (None, 259, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 259, 64)   │        512 │ waveform[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 259, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 129, 64)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 129, 64)   │          0 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 129, 128)  │     41,088 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 129, 128)  │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 64, 128)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64, 128)   │          0 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rr_features         │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 64, 256)   │     98,560 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        160 │ rr_features[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 256)   │      1,024 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        528 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 272)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │     17,472 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        195 │ dropout_2[0][0] 

 Total params: 160,435 (626.70 KB)

 Trainable params: 159,475 (622.95 KB)

 Non-trainable params: 960 (3.75 KB)

In [4]:
# ============================================================
# PHASE 4 — Cell 4: Prepare Dataset + Class Weights
# ============================================================
# Add channel dimension
X_train_c = X_train[..., np.newaxis].astype(np.float32)
X_val_c = X_val[..., np.newaxis].astype(np.float32)
X_test_c = X_test[..., np.newaxis].astype(np.float32)

R_train_f = R_train.astype(np.float32)
R_val_f = R_val.astype(np.float32)
R_test_f = R_test.astype(np.float32)

# Class weights (moderate - sqrt of balanced)
balanced_weights = compute_class_weight(
    'balanced', classes=np.unique(y_train), y=y_train
)
moderate_weights = np.sqrt(balanced_weights)
moderate_weights = moderate_weights / moderate_weights.mean()
class_weight_dict = {int(i): float(w) for i, w in enumerate(moderate_weights)}

print(f"✅ Class weights (moderate - sqrt):")
for i, name in enumerate(CLASS_NAMES):
    print(f"   {name}: {class_weight_dict[i]:.3f}")

# tf.data pipelines
BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, R, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices(({'waveform': X, 'rr_features': R}, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(X_train_c, R_train_f, y_train.astype(np.int64), training=True)
val_ds = make_ds(X_val_c, R_val_f, y_val.astype(np.int64))
test_ds = make_ds(X_test_c, R_test_f, y_test.astype(np.int64))

print(f"\n✅ Datasets ready")
print(f"   Train batches: {len(train_ds)}")
print(f"   Val batches:   {len(val_ds)}")
print(f"   Test batches:  {len(test_ds)}")
print(f"\n✅ Class weight dict: {class_weight_dict}")

✅ Class weights (moderate - sqrt):
   N/S: 0.390
   V: 1.433
   Q: 1.177

✅ Datasets ready
   Train batches: 350
   Val batches:   98
   Test batches:  403

✅ Class weight dict: {0: 0.39031330031133926, 1: 1.432869083095091, 2: 1.1768176165935693}


In [5]:
# ============================================================
# PHASE 4 — Cell 5: Train Model
# ============================================================
config = {
    'seed': 42,
    'batch_size': 128,
    'epochs': 30,
    'learning_rate': 5e-4,
    'focal_gamma': 2.0,
    'label_protocol': 'NS_V_Q',
    'split_protocol': 'standard_ds1_ds2',
    'early_stopping_patience': 6,
    'reduce_lr_patience': 3,
    'class_weights': class_weight_dict
}

print(f"Training config: {config}")

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=6, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, 'baseline_standard_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
    tf.keras.callbacks.CSVLogger(
        os.path.join(OUTPUTS_DIR, 'baseline_standard_history.csv')
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config['epochs'],
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print(f"\n✅ Training complete")
print(f"   Best val loss: {min(history.history['val_loss']):.4f}")
print(f"   Best val accuracy: {max(history.history['val_accuracy']):.4f}")

Training config: {'seed': 42, 'batch_size': 128, 'epochs': 30, 'learning_rate': 0.0005, 'focal_gamma': 2.0, 'label_protocol': 'NS_V_Q', 'split_protocol': 'standard_ds1_ds2', 'early_stopping_patience': 6, 'reduce_lr_patience': 3, 'class_weights': {0: 0.39031330031133926, 1: 1.432869083095091, 2: 1.1768176165935693}}
Epoch 1/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 30s 35ms/step - accuracy: 0.9443 - loss: 0.0681 - val_accuracy: 0.2013 - val_loss: 0.8862 - learning_rate: 5.0000e-04
Epoch 2/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9831 - loss: 0.0390 - val_accuracy: 0.7818 - val_loss: 1.0840 - learning_rate: 5.0000e-04
Epoch 3/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9875 - loss: 0.0303 - val_accuracy: 0.7966 - val_loss: 0.8733 - learning_rate: 5.0000e-04
Epoch 4/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9886 - loss: 0.0244 - val_accuracy: 0.7886 - val_loss: 1.2371 - learning_rate: 5.0000e-04
Epoch 5/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step -

In [6]:
# ============================================================
# PHASE 4-FIX — Cell 1: Load + 4-Class Labels
# ============================================================
# (Assumes X_train, y_train etc. already loaded from Cell 1)

# Re-load from Drive (fresh)
train = np.load(os.path.join(PROCESSED_DIR, 'train.npz'), allow_pickle=True)
val = np.load(os.path.join(PROCESSED_DIR, 'val.npz'), allow_pickle=True)
test = np.load(os.path.join(PROCESSED_DIR, 'test.npz'), allow_pickle=True)

X_train = train['X']; y_train_5c = train['y']; R_train = train['R']; rec_train = train['record_id']
X_val = val['X']; y_val_5c = val['y']; R_val = val['R']; rec_val = val['record_id']
X_test = test['X']; y_test_5c = test['y']; R_test = test['R']; rec_test = test['record_id']

# Apply 4-class labels (N, S, V, Q)
LABEL_MAP_4C = {0: 0, 1: 1, 2: 2, 4: 3}
CLASS_NAMES = ['N', 'S', 'V', 'Q']

def apply_4class(X, y, R, rec):
    mask = y != 3  # Drop F
    X2 = X[mask]
    y2 = y[mask].copy()
    R2 = R[mask]
    rec2 = rec[mask]
    new_y = np.zeros_like(y2)
    for old, new in LABEL_MAP_4C.items():
        new_y[y2 == old] = new
    return X2, new_y, R2, rec2

X_train, y_train, R_train, rec_train = apply_4class(X_train, y_train_5c, R_train, rec_train)
X_val, y_val, R_val, rec_val = apply_4class(X_val, y_val_5c, R_val, rec_val)
X_test, y_test, R_test, rec_test = apply_4class(X_test, y_test_5c, R_test, rec_test)

print(f"✅ 4-class data prepared")
print(f"Train: {dict(Counter(y_train))}")
print(f"Val:   {dict(Counter(y_val))}")
print(f"Test:  {dict(Counter(y_test))}")
print(f"Classes: {CLASS_NAMES}")

✅ 4-class data prepared
Train: {np.int64(0): 37176, np.int64(3): 4151, np.int64(1): 559, np.int64(2): 2800}
Val:   {np.int64(3): 2079, np.int64(2): 1053, np.int64(0): 8933, np.int64(1): 385}
Test:  {np.int64(0): 44483, np.int64(1): 1837, np.int64(2): 3382, np.int64(3): 1809}
Classes: ['N', 'S', 'V', 'Q']


In [7]:
# ============================================================
# PHASE 4-FIX — Cell 2: Build 4-class Model + Train
# ============================================================
def build_two_branch_4class(beat_len, n_rr_features=4, n_classes=4, seed=42):
    tf.keras.utils.set_random_seed(seed)
    reg = tf.keras.regularizers.l2(1e-4)

    # Waveform branch
    wave_in = layers.Input(shape=(beat_len, 1), name='waveform')
    x = layers.Conv1D(64, 7, activation='relu', padding='same', kernel_regularizer=reg)(wave_in)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, 5, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(256, 3, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    wave_feat = layers.GlobalAveragePooling1D()(x)

    # RR branch
    rr_in = layers.Input(shape=(n_rr_features,), name='rr_features')
    r = layers.Dense(32, activation='relu', kernel_regularizer=reg)(rr_in)
    r = layers.BatchNormalization()(r)
    r = layers.Dense(16, activation='relu', kernel_regularizer=reg)(r)

    # Merge
    merged = layers.Concatenate()([wave_feat, r])
    d = layers.Dense(64, activation='relu', kernel_regularizer=reg)(merged)
    d = layers.Dropout(0.4)(d)
    out = layers.Dense(n_classes, activation='softmax')(d)

    model = Model(inputs=[wave_in, rr_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss='sparse_categorical_crossentropy',  # ⚠️ NO FOCAL
        metrics=['accuracy']
    )
    return model


# Prepare data
X_train_c = X_train[..., np.newaxis].astype(np.float32)
X_val_c = X_val[..., np.newaxis].astype(np.float32)
X_test_c = X_test[..., np.newaxis].astype(np.float32)
R_train_f = R_train.astype(np.float32)
R_val_f = R_val.astype(np.float32)
R_test_f = R_test.astype(np.float32)

# Class weights (ONLY this, no focal)
cw_balanced = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_moderate = np.sqrt(cw_balanced)
cw_moderate = cw_moderate / cw_moderate.mean()
cw_dict = {int(i): float(w) for i, w in enumerate(cw_moderate)}
print(f"Class weights (moderate): {cw_dict}")

# Datasets
def make_ds(X, R, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices(({'waveform': X, 'rr_features': R}, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    return ds.batch(128).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train_c, R_train_f, y_train.astype(np.int64), training=True)
val_ds = make_ds(X_val_c, R_val_f, y_val.astype(np.int64))

# Build model
model = build_two_branch_4class(X_train_c.shape[1])
model.summary()

# Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, 'baseline_4class_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
    tf.keras.callbacks.CSVLogger(os.path.join(OUTPUTS_DIR, 'baseline_4class_history.csv'))
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=cw_dict,
    callbacks=callbacks,
    verbose=1
)

print(f"\n✅ Training complete")
print(f"   Best val loss: {min(history.history['val_loss']):.4f}")
print(f"   Best val accuracy: {max(history.history['val_accuracy']):.4f}")

Class weights (moderate): {0: 0.2533015935833029, 1: 2.065681441992995, 2: 0.9229756341128774, 3: 0.758041330310825}


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ waveform            │ (None, 259, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 259, 64)   │        512 │ waveform[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 259, 64)   │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 129, 64)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 129, 64)   │          0 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 129, 128)  │     41,088 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 129, 128)  │        512 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 64, 128)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64, 128)   │          0 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rr_features         │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 64, 256)   │     98,560 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │        160 │ rr_features[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 256)   │      1,024 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense_4[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 16)        │        528 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 272)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │     17,472 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 4)         │        260 │ dropout_5[0][0] 

 Total params: 160,500 (626.95 KB)

 Trainable params: 159,540 (623.20 KB)

 Non-trainable params: 960 (3.75 KB)

Epoch 1/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.9117 - loss: 0.1867 - val_accuracy: 0.7173 - val_loss: 1.1134 - learning_rate: 5.0000e-04
Epoch 2/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9668 - loss: 0.1025 - val_accuracy: 0.7933 - val_loss: 1.8282 - learning_rate: 5.0000e-04
Epoch 3/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9737 - loss: 0.0846 - val_accuracy: 0.7982 - val_loss: 2.0660 - learning_rate: 5.0000e-04
Epoch 4/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9781 - loss: 0.0759
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
350/350 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9777 - loss: 0.0749 - val_accuracy: 0.7923 - val_loss: 1.6112 - learning_rate: 5.0000e-04
Epoch 5/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9845 - loss: 0.0634 - val_accuracy: 0.7922 - val_loss: 1.7428 - learning_rate: 2.5000e-04
Epoch 6/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accurac

In [8]:
# ============================================================
# PHASE 4-FIX — Cell 2 (REVISED): Train with Fixed Config
# ============================================================
from sklearn.metrics import f1_score as f1_metric

class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, val_data, patience=10):
        super().__init__()
        self.val_data = val_data
        self.patience = patience
        self.best_f1 = 0.0
        self.best_weights = None
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        X_va, R_va, y_va = self.val_data
        probs = self.model.predict({'waveform': X_va, 'rr_features': R_va}, verbose=0)
        preds = np.argmax(probs, axis=1)
        macro_f1 = f1_metric(y_va, preds, average='macro', zero_division=0)
        logs['val_macro_f1'] = macro_f1
        print(f"  val_macro_f1: {macro_f1:.4f}")

        if macro_f1 > self.best_f1:
            self.best_f1 = macro_f1
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\nEarly stopping at epoch {epoch+1} (best F1: {self.best_f1:.4f})")
                self.model.stop_training = True
                self.model.set_weights(self.best_weights)


# Moderate manual weights (not extreme)
cw_dict = {
    0: 1.0,   # N
    1: 1.5,   # S (moderate boost)
    2: 1.2,   # V
    3: 1.0,   # Q
}
print(f"Class weights (moderate manual): {cw_dict}")

# Build model with lower LR
def build_two_branch_4class_v2(beat_len, n_rr_features=4, n_classes=4, seed=42):
    tf.keras.utils.set_random_seed(seed)
    reg = tf.keras.regularizers.l2(1e-4)

    wave_in = layers.Input(shape=(beat_len, 1), name='waveform')
    x = layers.Conv1D(64, 7, activation='relu', padding='same', kernel_regularizer=reg)(wave_in)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv1D(128, 5, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv1D(256, 3, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    wave_feat = layers.GlobalAveragePooling1D()(x)

    rr_in = layers.Input(shape=(n_rr_features,), name='rr_features')
    r = layers.Dense(32, activation='relu', kernel_regularizer=reg)(rr_in)
    r = layers.BatchNormalization()(r)
    r = layers.Dense(16, activation='relu', kernel_regularizer=reg)(r)

    merged = layers.Concatenate()([wave_feat, r])
    d = layers.Dense(64, activation='relu', kernel_regularizer=reg)(merged)
    d = layers.Dropout(0.4)(d)
    out = layers.Dense(n_classes, activation='softmax')(d)

    model = Model(inputs=[wave_in, rr_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),  # Lower LR
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


model = build_two_branch_4class_v2(X_train_c.shape[1], seed=42)

callbacks = [
    MacroF1Callback(
        val_data=(X_val_c, R_val_f, y_val.astype(np.int64)),
        patience=10
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, 'baseline_4class_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
    tf.keras.callbacks.CSVLogger(os.path.join(OUTPUTS_DIR, 'baseline_4class_history.csv'))
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=cw_dict,
    callbacks=callbacks,
    verbose=1
)

print(f"\n✅ Training complete")
print(f"   Best val loss: {min(history.history['val_loss']):.4f}")
print(f"   Best val accuracy: {max(history.history['val_accuracy']):.4f}")

Class weights (moderate manual): {0: 1.0, 1: 1.5, 2: 1.2, 3: 1.0}
Epoch 1/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7008 - loss: 0.8879  val_macro_f1: 0.2089
350/350 ━━━━━━━━━━━━━━━━━━━━ 29s 50ms/step - accuracy: 0.8252 - loss: 0.6396 - val_accuracy: 0.7175 - val_loss: 2.8636 - val_macro_f1: 0.2089 - learning_rate: 1.0000e-04
Epoch 2/30
346/350 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9326 - loss: 0.3564  val_macro_f1: 0.2099
350/350 ━━━━━━━━━━━━━━━━━━━━ 23s 18ms/step - accuracy: 0.9413 - loss: 0.3191 - val_accuracy: 0.7176 - val_loss: 2.3739 - val_macro_f1: 0.2099 - learning_rate: 1.0000e-04
Epoch 3/30
349/350 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9561 - loss: 0.2413  val_macro_f1: 0.3978
350/350 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9586 - loss: 0.2302 - val_accuracy: 0.7687 - val_loss: 2.1655 - val_macro_f1: 0.3978 - learning_rate: 1.0000e-04
Epoch 4/30
346/350 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9662 - loss: 0.1916  val_macro_

In [9]:
# ============================================================
# THE ONE TRUE CV CELL — 4-class (N,S,V,Q) + RR features
# class_weight = compute_class_weight('balanced') x config multiplier
# NO focal loss. NO manual/arbitrary weights.
# ============================================================
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter
import numpy as np

candidate_configs = [
    {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0},
    {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8},
    {'N': 1.0, 'S': 2.0, 'V': 0.8, 'Q': 0.8},
    {'N': 1.0, 'S': 1.5, 'V': 1.0, 'Q': 1.0},
]

oof_results = {}

for cfg_i, cfg in enumerate(candidate_configs):
    print(f'\n===== Config {cfg_i+1}/{len(candidate_configs)}: {cfg} =====')
    oof_preds = np.full(len(y_pool), -1, dtype=np.int64)

    for fold_i, (tr_idx, va_idx) in enumerate(fold_splits):
        X_tr_raw, y_tr, R_tr_raw = X_pool[tr_idx], y_pool[tr_idx], R_pool[tr_idx]
        X_va_raw, y_va, R_va_raw = X_pool[va_idx], y_pool[va_idx], R_pool[va_idx]

        X_tr = per_beat_normalize(X_tr_raw)
        X_va = per_beat_normalize(X_va_raw)

        fold_rr_mean = R_tr_raw.mean(axis=0)
        fold_rr_std  = R_tr_raw.std(axis=0) + 1e-8
        R_tr = (R_tr_raw - fold_rr_mean) / fold_rr_std
        R_va = (R_va_raw - fold_rr_mean) / fold_rr_std

        combined_tr = np.concatenate([X_tr, R_tr], axis=1)
        counts = Counter(y_tr)
        majority = max(counts.values())
        strat = {}
        for cls, cnt in counts.items():
            if cnt < majority:
                strat[cls] = max(min(int(majority * 0.25), majority), cnt)
        min_cnt = min(counts.values())
        k = min(5, max(1, min_cnt - 1))
        sm = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=strat)
        combined_res, y_tr_res = sm.fit_resample(combined_tr, y_tr)

        beat_len_fold = X_tr.shape[1]
        X_tr_res = combined_res[:, :beat_len_fold]
        R_tr_res = combined_res[:, beat_len_fold:]

        train_ds_fold = make_ds(add_channel(X_tr_res), R_tr_res.astype(np.float32),
                                 y_tr_res.astype(np.int64), training=True)

        classes_present_fold = np.unique(y_tr_res)
        base_w = compute_class_weight('balanced', classes=classes_present_fold, y=y_tr_res)
        base_w_dict = dict(zip(classes_present_fold.tolist(), base_w.tolist()))
        cw = {i: base_w_dict[i] * cfg[CLASS_NAMES[i]] for i in classes_present_fold}

        m = build_two_branch_model(beat_len_fold, seed=42)
        m.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # NO focal loss
        m.fit(train_ds_fold, epochs=30,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=6, restore_best_weights=True)],
              class_weight=cw, verbose=0)

        probs = m.predict({'waveform': add_channel(X_va), 'rr_features': R_va.astype(np.float32)}, verbose=0)
        preds = np.argmax(probs, axis=1)
        oof_preds[va_idx] = preds
        print(f'  Fold {fold_i+1} done ({len(va_idx)} beats)')

    assert (oof_preds == -1).sum() == 0
    oof_f1 = f1_score(y_pool, oof_preds, average='macro')
    oof_results[cfg_i] = (oof_f1, oof_preds.copy())
    print(f'Config {cfg} -> OOF macro-F1 (full 25-patient pool): {oof_f1:.4f}')

print('\n\n===== FINAL OOF SUMMARY =====')
for cfg_i, cfg in enumerate(candidate_configs):
    print(f'{cfg}: OOF macro-F1={oof_results[cfg_i][0]:.4f}')

best_cfg_i = max(oof_results, key=lambda i: oof_results[i][0])
best_config = candidate_configs[best_cfg_i]
best_oof_preds = oof_results[best_cfg_i][1]
print(f'\nBest config (by OOF macro-F1 on full 25-patient pool): {best_config}')

print('\n===== Classification report (OOF, best config, full pool) =====')
print(classification_report(y_pool, best_oof_preds, target_names=CLASS_NAMES, digits=4, zero_division=0))


===== Config 1/4: {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0} =====


NameError: name 'y_pool' is not defined

In [10]:
# ============================================================
# FULL RESET — rebuild everything needed for the OOF-CV cell
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import tensorflow as tf
from collections import Counter
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_class_weight

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')

def load_split(name):
    d = np.load(os.path.join(PROCESSED_DIR, f'{name}.npz'))
    return d['X'], d['y'], d['R'], d['record_id']

X_train, y_train, R_train, rec_train = load_split('train')
X_val,   y_val,   R_val,   rec_val   = load_split('val')

CLASS_NAMES = ['N', 'S', 'V', 'Q']

def drop_F_and_remap(X, y, R, rec):
    mask = y != 3
    X2, y2, R2, rec2 = X[mask], y[mask].copy(), R[mask], rec[mask]
    y2[y2 == 4] = 3
    return X2, y2, R2, rec2

X_train, y_train, R_train, rec_train = drop_F_and_remap(X_train, y_train, R_train, rec_train)
X_val,   y_val,   R_val,   rec_val   = drop_F_and_remap(X_val,   y_val,   R_val,   rec_val)

# Pool = train + val (25 patients) for CV; test.npz stays untouched, not loaded here
X_pool = np.concatenate([X_train, X_val], axis=0)
y_pool = np.concatenate([y_train, y_val], axis=0)
R_pool = np.concatenate([R_train, R_val], axis=0)
groups_pool = np.concatenate([rec_train, rec_val], axis=0)

print('Pool:', X_pool.shape, '| patients:', len(np.unique(groups_pool)), '| classes:', Counter(y_pool))

# ---- Helper functions ----
def per_beat_normalize(X, eps=1e-8):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    return (X - m) / (s + eps)

def add_channel(X):
    return X[..., np.newaxis].astype(np.float32)

BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, R, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices(({'waveform': X, 'rr_features': R}, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def build_two_branch_model(beat_len, n_rr_features=4, n_classes=4, seed=42):
    tf.keras.utils.set_random_seed(seed)
    reg = tf.keras.regularizers.l2(1e-4)

    wave_in = tf.keras.layers.Input(shape=(beat_len, 1), name='waveform')
    x = tf.keras.layers.Conv1D(32, 7, activation='relu', padding='same', kernel_regularizer=reg)(wave_in)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(64, 5, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(128, 3, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    wave_feat = tf.keras.layers.GlobalAveragePooling1D()(x)

    rr_in = tf.keras.layers.Input(shape=(n_rr_features,), name='rr_features')
    r = tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=reg)(rr_in)
    r = tf.keras.layers.BatchNormalization()(r)
    r = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg)(r)

    merged = tf.keras.layers.Concatenate()([wave_feat, r])
    d = tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=reg)(merged)
    d = tf.keras.layers.Dropout(0.4)(d)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(d)

    model = tf.keras.Model(inputs=[wave_in, rr_in], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ---- Fold splits ----
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
fold_splits = list(gkf.split(X_pool, y_pool, groups=groups_pool))
for i, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'Fold {i+1}: train={len(tr_idx)}, val={len(va_idx)}, val_patients={sorted(set(groups_pool[va_idx]))}')

print('\n✅ Reset complete. Everything needed for the OOF-CV cell is now defined.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pool: (57136, 259) | patients: 25 | classes: Counter({np.int64(0): 46109, np.int64(3): 6230, np.int64(2): 3853, np.int64(1): 944})
Fold 1: train=45663, val=11473, val_patients=[np.str_('107'), np.str_('115'), np.str_('116'), np.str_('124'), np.str_('215')]
Fold 2: train=45611, val=11525, val_patients=[np.str_('104'), np.str_('122'), np.str_('201'), np.str_('207'), np.str_('209')]
Fold 3: train=45695, val=11441, val_patients=[np.str_('102'), np.str_('108'), np.str_('109'), np.str_('119'), np.str_('203')]
Fold 4: train=45789, val=11347, val_patients=[np.str_('101'), np.str_('112'), np.str_('205'), np.str_('220'), np.str_('230')]
Fold 5: train=45786, val=11350, val_patients=[np.str_('106'), np.str_('114'), np.str_('118'), np.str_('208'), np.str_('223')]

✅ Reset complete. Everything needed for the OOF-CV cell is now defined.


In [11]:
# ============================================================
# THE ONE TRUE CV CELL — 4-class (N,S,V,Q) + RR features
# class_weight = compute_class_weight('balanced') x config multiplier
# NO focal loss. NO manual/arbitrary weights.
# ============================================================
candidate_configs = [
    {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0},
    {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8},
    {'N': 1.0, 'S': 2.0, 'V': 0.8, 'Q': 0.8},
    {'N': 1.0, 'S': 1.5, 'V': 1.0, 'Q': 1.0},
]

oof_results = {}

for cfg_i, cfg in enumerate(candidate_configs):
    print(f'\n===== Config {cfg_i+1}/{len(candidate_configs)}: {cfg} =====')
    oof_preds = np.full(len(y_pool), -1, dtype=np.int64)

    for fold_i, (tr_idx, va_idx) in enumerate(fold_splits):
        X_tr_raw, y_tr, R_tr_raw = X_pool[tr_idx], y_pool[tr_idx], R_pool[tr_idx]
        X_va_raw, y_va, R_va_raw = X_pool[va_idx], y_pool[va_idx], R_pool[va_idx]

        X_tr = per_beat_normalize(X_tr_raw)
        X_va = per_beat_normalize(X_va_raw)

        fold_rr_mean = R_tr_raw.mean(axis=0)
        fold_rr_std  = R_tr_raw.std(axis=0) + 1e-8
        R_tr = (R_tr_raw - fold_rr_mean) / fold_rr_std
        R_va = (R_va_raw - fold_rr_mean) / fold_rr_std

        combined_tr = np.concatenate([X_tr, R_tr], axis=1)
        counts = Counter(y_tr)
        majority = max(counts.values())
        strat = {}
        for cls, cnt in counts.items():
            if cnt < majority:
                strat[cls] = max(min(int(majority * 0.25), majority), cnt)
        min_cnt = min(counts.values())
        k = min(5, max(1, min_cnt - 1))
        sm = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=strat)
        combined_res, y_tr_res = sm.fit_resample(combined_tr, y_tr)

        beat_len_fold = X_tr.shape[1]
        X_tr_res = combined_res[:, :beat_len_fold]
        R_tr_res = combined_res[:, beat_len_fold:]

        train_ds_fold = make_ds(add_channel(X_tr_res), R_tr_res.astype(np.float32),
                                 y_tr_res.astype(np.int64), training=True)

        classes_present_fold = np.unique(y_tr_res)
        base_w = compute_class_weight('balanced', classes=classes_present_fold, y=y_tr_res)
        base_w_dict = dict(zip(classes_present_fold.tolist(), base_w.tolist()))
        cw = {i: base_w_dict[i] * cfg[CLASS_NAMES[i]] for i in classes_present_fold}

        m = build_two_branch_model(beat_len_fold, seed=42)
        m.fit(train_ds_fold, epochs=30,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=6, restore_best_weights=True)],
              class_weight=cw, verbose=0)

        probs = m.predict({'waveform': add_channel(X_va), 'rr_features': R_va.astype(np.float32)}, verbose=0)
        preds = np.argmax(probs, axis=1)
        oof_preds[va_idx] = preds
        print(f'  Fold {fold_i+1} done ({len(va_idx)} beats)')

    assert (oof_preds == -1).sum() == 0
    oof_f1 = f1_score(y_pool, oof_preds, average='macro')
    oof_results[cfg_i] = (oof_f1, oof_preds.copy())
    print(f'Config {cfg} -> OOF macro-F1 (full 25-patient pool): {oof_f1:.4f}')

print('\n\n===== FINAL OOF SUMMARY =====')
for cfg_i, cfg in enumerate(candidate_configs):
    print(f'{cfg}: OOF macro-F1={oof_results[cfg_i][0]:.4f}')

best_cfg_i = max(oof_results, key=lambda i: oof_results[i][0])
best_config = candidate_configs[best_cfg_i]
best_oof_preds = oof_results[best_cfg_i][1]
print(f'\nBest config (by OOF macro-F1 on full 25-patient pool): {best_config}')

print('\n===== Classification report (OOF, best config, full pool) =====')
print(classification_report(y_pool, best_oof_preds, target_names=CLASS_NAMES, digits=4, zero_division=0))


===== Config 1/4: {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0} =====
  Fold 1 done (11473 beats)
  Fold 2 done (11525 beats)
  Fold 3 done (11441 beats)
  Fold 4 done (11347 beats)
  Fold 5 done (11350 beats)
Config {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0} -> OOF macro-F1 (full 25-patient pool): 0.6304

===== Config 2/4: {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8} =====
  Fold 1 done (11473 beats)
  Fold 2 done (11525 beats)
  Fold 3 done (11441 beats)
  Fold 4 done (11347 beats)
  Fold 5 done (11350 beats)
Config {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8} -> OOF macro-F1 (full 25-patient pool): 0.6020

===== Config 3/4: {'N': 1.0, 'S': 2.0, 'V': 0.8, 'Q': 0.8} =====
  Fold 1 done (11473 beats)
  Fold 2 done (11525 beats)
  Fold 3 done (11441 beats)
  Fold 4 done (11347 beats)
  Fold 5 done (11350 beats)
Config {'N': 1.0, 'S': 2.0, 'V': 0.8, 'Q': 0.8} -> OOF macro-F1 (full 25-patient pool): 0.5793

===== Config 4/4: {'N': 1.0, 'S': 1.5, 'V': 1.0, 'Q': 1.0} =====
  Fold 1 done (11473 beats)
  F

In [12]:
# ============================================================
# FINAL Baseline A: train on FULL 25-patient pool (best config),
# evaluate ONCE on the untouched TEST set (23 patients)
# ============================================================

# Load test set (loaded fresh here for the first time in this notebook)
X_test, y_test, R_test, rec_test = load_split('test')
X_test, y_test, R_test, rec_test = drop_F_and_remap(X_test, y_test, R_test, rec_test)
print('Test:', X_test.shape, Counter(y_test))

# Normalize
X_pool_n = per_beat_normalize(X_pool)
X_test_n = per_beat_normalize(X_test)

pool_rr_mean = R_pool.mean(axis=0)
pool_rr_std  = R_pool.std(axis=0) + 1e-8
R_pool_n = (R_pool - pool_rr_mean) / pool_rr_std
R_test_n = (R_test - pool_rr_mean) / pool_rr_std

# Capped SMOTE on the full pool
combined_pool = np.concatenate([X_pool_n, R_pool_n], axis=1)
counts = Counter(y_pool)
majority = max(counts.values())
strat = {}
for cls, cnt in counts.items():
    if cnt < majority:
        strat[cls] = max(min(int(majority * 0.25), majority), cnt)
min_cnt = min(counts.values())
k = min(5, max(1, min_cnt - 1))
sm = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=strat)
combined_res, y_pool_res = sm.fit_resample(combined_pool, y_pool)

beat_len_final = X_pool_n.shape[1]
X_pool_res = combined_res[:, :beat_len_final]
R_pool_res = combined_res[:, beat_len_final:]

train_ds_final = make_ds(add_channel(X_pool_res), R_pool_res.astype(np.float32),
                          y_pool_res.astype(np.int64), training=True)

classes_present = np.unique(y_pool_res)
base_w = compute_class_weight('balanced', classes=classes_present, y=y_pool_res)
final_class_weight = dict(zip(classes_present.tolist(), base_w.tolist()))  # best_config was uniform, so this IS the final weight
print('Final class weights:', dict(zip(CLASS_NAMES, base_w.tolist())))

final_model = build_two_branch_model(beat_len_final, seed=42)
final_model.fit(train_ds_final, epochs=30,
                 callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=6, restore_best_weights=True)],
                 class_weight=final_class_weight, verbose=1)

# ===== THE ONE, OFFICIAL, LOCKED TEST EVALUATION =====
y_pred_probs = final_model.predict({'waveform': add_channel(X_test_n), 'rr_features': R_test_n.astype(np.float32)}, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print('\n===== 🔒 LOCKED TEST RESULTS — Baseline A (4-class + RR, CV-validated) =====\n')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))
print('Macro-F1:', f1_score(y_test, y_pred, average='macro'))
print('Weighted-F1:', f1_score(y_test, y_pred, average='weighted'))

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion matrix:')
print('     ', '  '.join(f'{c:>5s}' for c in CLASS_NAMES))
for i, row in enumerate(cm):
    print(f'{CLASS_NAMES[i]:>5s}', '  '.join(f'{v:5d}' for v in row))

y_test_onehot = tf.keras.utils.to_categorical(y_test, num_classes=4)
print('\nPer-class AUROC / AUPRC:')
for i, cname in enumerate(CLASS_NAMES):
    auroc = roc_auc_score(y_test_onehot[:, i], y_pred_probs[:, i])
    auprc = average_precision_score(y_test_onehot[:, i], y_pred_probs[:, i])
    print(f'  {cname}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}')

final_model.save(os.path.join(PROJECT_DIR, 'models', 'baseline_a_final_locked.keras'))
print('\nModel saved. THIS IS THE OFFICIAL BASELINE A RESULT — do not re-tune after this.')

Test: (51511, 259) Counter({np.int64(0): 44483, np.int64(2): 3382, np.int64(1): 1837, np.int64(3): 1809})
Final class weights: {'N': 0.43749593354876487, 'S': 1.7500216882102888, 'V': 1.7500216882102888, 'Q': 1.7500216882102888}
Epoch 1/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - accuracy: 0.8877 - loss: 0.3216
Epoch 2/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9604 - loss: 0.1298
Epoch 3/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9702 - loss: 0.1019
Epoch 4/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.9750 - loss: 0.0897
Epoch 5/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9782 - loss: 0.0797
Epoch 6/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9814 - loss: 0.0720
Epoch 7/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9825 - loss: 0.0681
Epoch 8/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9846 - loss: 0.0635
Epoch 9/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9855 - loss: 0